In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from geopy.distance import geodesic
from scipy.signal import medfilt

def calculate_time_differences_manually(df, datetime_column='datetime', output_column='time_diff'):
    """
    Calculates the time difference (in seconds) between consecutive rows.
    """
    df[datetime_column] = pd.to_datetime(df[datetime_column])
    df = df.sort_values(datetime_column).drop_duplicates(subset=[datetime_column]).reset_index(drop=True)
    datetimes = df[datetime_column].tolist()
    time_diffs = [float('nan')]
    for i in range(1, len(datetimes)):
        delta = datetimes[i] - datetimes[i-1]
        time_diffs.append(delta.total_seconds())
    df[output_column] = time_diffs
    return df

def check_raw_gps_data(df):
    """
    Prints basic diagnostics of the raw GPS data (lat/lon differences and time diff statistics).
    """
    if not pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['datetime'] = pd.to_datetime(df['datetime'])
    else:
        print("Datetime column is already in datetime64 format; proceeding as-is.")
    
    df['lat_diff'] = df['lat'].diff().abs()
    df['lon_diff'] = df['lon'].diff().abs()
    
    print("\nFirst 20 rows with differences:")
    # Assumes that calculate_time_differences_manually() has been applied
    print(df[['datetime', 'lat_diff', 'lon_diff', 'time_diff']].iloc[:20])
    print("\nTime Difference Statistics (seconds):")
    print(df['time_diff'].describe())
    
    print("\nPotential Latitude Outliers:")
    print(df[df['lat_diff'] > df['lat_diff'].mean() + 3 * df['lat_diff'].std()][['lat', 'lat_diff', 'datetime']])
    print("\nPotential Longitude Outliers:")
    print(df[df['lon_diff'] > df['lon_diff'].mean() + 3 * df['lon_diff'].std()][['lon', 'lon_diff', 'datetime']])
    print("\nPotential Time Outliers (time differences < 0.5 sec):")
    print(df[df['time_diff'] < 0.5][['datetime', 'time_diff']])
    
    return df

def clean_trajectory(df):
    """
    Performs cleaning on a raw trajectory:
      - Checks that there are enough data points
      - Computes time differences (if not already computed)
      - Computes the next coordinates, geodesic distances, and speeds
      - Applies median filtering to speed
      - Discards trajectories with unrealistic speed profiles
      - Computes bearing and bearing changes
    """
    # Discard if too few rows
    if len(df) < 10:
        print(f"Data discarded: insufficient rows ({len(df)} rows).")
        return pd.DataFrame()
    
    # Ensure time differences are available
    if 'time_diff' not in df.columns:
        df = calculate_time_differences_manually(df)
    
    # Compute next coordinates for distance and bearing calculation
    df['next_lat'] = df['lat'].shift(-1)
    df['next_lon'] = df['lon'].shift(-1)
    
    # Calculate geodesic distance (meters) between consecutive points
    df['distance'] = df.apply(
        lambda row: geodesic((row['lat'], row['lon']), (row['next_lat'], row['next_lon'])).meters
        if pd.notna(row['next_lat']) else np.nan,
        axis=1
    )
    
    # Compute speed (m/s) and smooth using median filter (kernel size=5)
    df['speed'] = df['distance'] / df['time_diff']
    df['speed'] = medfilt(df['speed'], kernel_size=5)
    
    # Basic quality control: reject trajectories with too high mean speed
    mean_speed = df['speed'].mean()
    if mean_speed > 15:
        print(f"Data discarded: mean speed too high ({mean_speed:.2f} m/s).")
        return pd.DataFrame()
    
    # Reject trajectories where more than 50% of rows have zero speed
    zero_speed_count = (df['speed'] == 0).sum()
    if zero_speed_count > len(df) * 0.5:
        print(f"Data discarded: more than 50% of rows have zero speed ({zero_speed_count} rows).")
        return pd.DataFrame()
    
    # Compute bearing between consecutive points and the absolute change in bearing
    df['bearing'] = np.where(
        pd.notna(df['next_lat']),
        compute_bearing(df['lat'], df['lon'], df['next_lat'], df['next_lon']),
        np.nan
    )
    df['bearing_change'] = df['bearing'].diff().abs().fillna(0)
    
    print(f"Data retained: {len(df)} rows after cleaning.")
    return df

# Helper: Compute bearing (used in enrichment)
def compute_bearing(lat1, lon1, lat2, lon2):
    """
    Compute bearing between two GPS coordinates.
    """
    dlon = np.radians(lon2 - lon1)
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.degrees(np.arctan2(y, x)) % 360

In [2]:
from geopy.geocoders import Nominatim

def get_location_name(lat, lon):
    """
    Enriches a GPS coordinate by retrieving a human-readable location name.
    """
    geolocator = Nominatim(user_agent="geoenrichment")
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True)
        return location.address
    except Exception as e:
        return "Unknown Location"

def minimal_angle_diff(diff):
    """Compute the minimal absolute difference between two angles (in degrees)."""
    diff = abs(diff) % 360
    return diff if diff <= 180 else 360 - diff

def compute_enriched_metrics(df):
    """
    Computes enriched trip metrics:
      - Step distances and overall distance (km)
      - Raw speeds with mode-based speed capping and outlier filtering
      - Horizontal acceleration (m/s²)
      - Turning metrics (bearing change, turn rate, etc.)
      - Altitude-related metrics (if 'altitude' exists):
          * avg_altitude (m)
          * total_altitude_gain (m)
          * total_altitude_loss (m)
          * avg_vertical_acceleration (m/s²)
    """
    # Ensure time differences are available
    if 'time_diff' not in df.columns:
        df = calculate_time_differences_manually(df)
    
    # Compute next coordinates and step distance
    df['next_lat'] = df['lat'].shift(-1)
    df['next_lon'] = df['lon'].shift(-1)
    df = df.dropna(subset=['lat', 'lon', 'next_lat', 'next_lon'])
    df['step_distance'] = df.apply(
        lambda row: geodesic((row['lat'], row['lon']), (row['next_lat'], row['next_lon'])).meters,
        axis=1
    )
    total_distance = df['step_distance'].sum() / 1000  # in km
    
    # Compute raw speed (m/s)
    df['speed'] = df['step_distance'] / df['time_diff']
    
    # Apply mode-based speed cap
    if 'transport_mode' in df.columns and not df['transport_mode'].isna().all():
        mode = df['transport_mode'].mode()[0] if not df['transport_mode'].mode().empty else 'unknown'
    else:
        mode = 'unknown'
    
    speed_cap_mps = {
        'walk': 2.78,  # 10 km/h
        'bike': 6.94,  # 25 km/h
        'bus': 11.11,  # 40 km/h
        'car': 13.89,  # 50 km/h
        'taxi': 13.89, # 50 km/h
        'unknown': 13.89
    }.get(mode.lower(), 6.94)
    
    accel_cap_mps2 = {
        'walk': 2.0,
        'bike': 3.0,
        'bus': 3.5,
        'car': 4.0,
        'taxi': 4.0,
        'unknown': 4.0
    }.get(mode.lower(), 4.0)
    
    # Outlier filtering using IQR method for speed
    original_speed = df['speed'].copy()
    Q1 = df['speed'].quantile(0.25)
    Q3 = df['speed'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    final_speed_cap = min(upper_bound, speed_cap_mps)
    outlier_count = (original_speed > final_speed_cap).sum()
    print("Outlier count before clipping:", outlier_count)
    
    df['speed'] = df['speed'].clip(upper=final_speed_cap)
    df['speed'] = medfilt(df['speed'], kernel_size=5)
    df['speed_kmh'] = df['speed'] * 3.6  # Convert m/s to km/h
    
    # Compute horizontal acceleration (m/s²)
    df['acceleration'] = df['speed'].diff() / df['time_diff']
    df['acceleration'] = df['acceleration'].fillna(0)
    
    # Turning metrics: bearing and bearing change
    df['bearing'] = df.apply(
        lambda row: compute_bearing(row['lat'], row['lon'], row['next_lat'], row['next_lon']),
        axis=1
    )
    df['bearing_change'] = df['bearing'].diff().apply(minimal_angle_diff)
    turn_threshold = 30  # degrees
    num_turns = (df['bearing_change'] >= turn_threshold).sum()
    duration_minutes = df['time_diff'].sum() / 60
    turn_rate = num_turns / duration_minutes if duration_minutes > 0 else 0
    avg_turn_angle = df['bearing_change'].mean()
    turn_angle_std = df['bearing_change'].std()
    
    # Altitude-related metrics (if available)
    if 'altitude' in df.columns:
        df['altitude'] = pd.to_numeric(df['altitude'], errors='coerce')
        # Replace outlier altitude values with 0.
        # Here we assume a realistic range of 0 to 300 m; adjust these bounds as needed.
        df.loc[(df['altitude'] < 0) | (df['altitude'] > 300), 'altitude'] = 0
        
        # Sort by datetime to ensure correct time sequence
        df = df.sort_values('datetime').reset_index(drop=True)
        df['alt_diff'] = df['altitude'].diff()
        total_altitude_gain = df['alt_diff'][df['alt_diff'] > 0].sum() if not df['alt_diff'][df['alt_diff'] > 0].empty else 0
        total_altitude_loss = abs(df['alt_diff'][df['alt_diff'] < 0]).sum() if not df['alt_diff'][df['alt_diff'] < 0].empty else 0
        avg_altitude = df['altitude'].mean()
        # Compute vertical speed (m/s) and vertical acceleration
        df['vertical_speed'] = df['alt_diff'] / df['time_diff']
        df['vertical_acceleration'] = df['vertical_speed'].diff() / df['time_diff'].shift(-1)
        avg_vertical_acceleration = df['vertical_acceleration'].abs().mean()
    else:
        avg_altitude = total_altitude_gain = total_altitude_loss = avg_vertical_acceleration = None
    
    metrics = {
        "total_distance": total_distance,
        "max_speed": df['speed_kmh'].max(),
        "min_speed": df['speed_kmh'].min(),
        "speed_std": df['speed_kmh'].std(),
        "avg_speed": df['speed_kmh'].mean(),
        "avg_acceleration": df['acceleration'].mean(),
        "max_acceleration": df['acceleration'].max(),
        "acceleration_std": df['acceleration'].std(),
        "num_turns": int(num_turns),
        "turn_rate": turn_rate,
        "avg_turn_angle": avg_turn_angle,
        "turn_angle_std": turn_angle_std,
        "avg_bearing_change": avg_turn_angle,
        # Altitude metrics
        "avg_altitude": avg_altitude,
        "total_altitude_gain": total_altitude_gain,
        "total_altitude_loss": total_altitude_loss,
        "avg_vertical_acceleration": avg_vertical_acceleration
    }
    return metrics

def generate_trip_description(df, metrics):
    """
    Generates a human-readable summary of the trip using enriched metrics.
    Also enriches the trip by retrieving start and end location names.
    Now includes altitude information if available.
    """
    if df.empty:
        return "No valid data for this trip.", "Unknown"
    
    start = df.iloc[0]
    end = df.iloc[-1]
    
    start_location = get_location_name(start['lat'], start['lon'])
    end_location = get_location_name(end['lat'], end['lon'])
    
    if 'transport_mode' in df.columns and not df['transport_mode'].isna().all():
        mode_series = df['transport_mode']
        mode_value = mode_series.mode()[0]
        mode_count = (mode_series == mode_value).sum()
        total_points = len(mode_series)
        threshold = 0.6
        if (mode_count / total_points) >= threshold:
            transport_mode = mode_value
        else:
            transport_mode = "Mixed"
    else:
        transport_mode = "Unknown"
    
    description = f"""
    Trip Summary:
    - Start: {start['datetime'].strftime('%Y-%m-%d %H:%M:%S')} at {start_location}
    - End: {end['datetime'].strftime('%Y-%m-%d %H:%M:%S')} at {end_location}
    - Duration: {end['datetime'] - start['datetime']}
    - Distance: {metrics['total_distance']:.2f} km
    - Average Speed: {metrics['avg_speed']:.2f} km/h
    - Average Bearing Change: {metrics['avg_bearing_change']:.2f}°
    - Max Speed: {metrics['max_speed']:.2f} km/h
    - Min Speed: {metrics['min_speed']:.2f} km/h
    - Speed Variability: {metrics['speed_std']:.2f} km/h
    - Average Acceleration: {metrics['avg_acceleration']:.2f} m/s²
    - Max Acceleration: {metrics['max_acceleration']:.2f} m/s²
    - Number of Turns: {metrics['num_turns']}
    - Turn Rate: {metrics['turn_rate']:.2f} turns/min
    - Average Turn Angle: {metrics['avg_turn_angle']:.2f}°
    - Turn Angle Variability: {metrics['turn_angle_std']:.2f}°
    """
    # If altitude data is available, add it to the description.
    if metrics['avg_altitude'] is not None:
        description += f"""- Average Altitude: {metrics['avg_altitude']:.2f} m
    - Total Altitude Gain: {metrics['total_altitude_gain']:.2f} m
    - Total Altitude Loss: {metrics['total_altitude_loss']:.2f} m
    - Average Vertical Acceleration: {metrics['avg_vertical_acceleration']:.2f} m/s²
    """
    description += f"""- Transport Mode: {transport_mode}"""
    
    return description, transport_mode

In [ ]:
import os
import glob
import geopandas as gpd
import pandas as pd
import warnings
from geopy.distance import geodesic
from scipy.signal import medfilt
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Folder where the cleaned sub-trajectories are stored.
folder_path = "Sub_Trajectories_Cleaned"
geojson_files = glob.glob(os.path.join(folder_path, "**/*.geojson"), recursive=True)
print(f"Number of GeoJSON files found: {len(geojson_files)}")

# Initialize accumulators for global analysis
total_rows_global = 0
extreme_outlier_rows_global = 0
very_extreme_outlier_rows_global = 0
regular_capped_rows_global = 0

# List to store trip-level metrics
trip_data = []
# counter = 0
for file in geojson_files:
    try:
        # counter += 1
        # if counter == 20:
        #     break
        # Read the file into a GeoDataFrame.
        gdf = gpd.read_file(file)
        if gdf.empty:
            print(f"File {file} is empty. Skipping.")
            continue

        # Clean the trajectory using your function.
        gdf_clean = clean_trajectory(gdf)
        if gdf_clean.empty:
            print(f"File {file} did not pass cleaning. Skipping.")
            continue

        # Ensure time differences are computed.
        gdf_clean = calculate_time_differences_manually(gdf_clean)
        if not pd.api.types.is_datetime64_any_dtype(gdf_clean['datetime']):
            gdf_clean['datetime'] = pd.to_datetime(gdf_clean['datetime'])
        gdf_clean = gdf_clean.sort_values('datetime').reset_index(drop=True)

        # Compute necessary step distances for speed calculation
        gdf_clean['next_lat'] = gdf_clean['lat'].shift(-1)
        gdf_clean['next_lon'] = gdf_clean['lon'].shift(-1)
        gdf_clean = gdf_clean.dropna(subset=['lat', 'lon', 'next_lat', 'next_lon'])
        gdf_clean['step_distance'] = gdf_clean.apply(
            lambda row: geodesic((row['lat'], row['lon']), (row['next_lat'], row['next_lon'])).meters,
            axis=1
        )
        
        # Compute raw speeds
        gdf_clean['speed'] = gdf_clean['step_distance'] / gdf_clean['time_diff']

        # Determine mode for capping
        if 'transport_mode' in gdf_clean.columns and not gdf_clean['transport_mode'].isna().all():
            mode = gdf_clean['transport_mode'].mode()[0] if not gdf_clean['transport_mode'].mode().empty else 'unknown'
        else:
            mode = 'unknown'
        
        # Compute the maximum speed for the trip
        trip_max_speed = gdf_clean['speed'].max()
            
        speed_cap_mps = {
            'walk': 2.78,   # 10 km/h
            'bike': 6.94,   # 25 km/h
            'bus': 11.11,   # 40 km/h
            'car': 13.89,   # 50 km/h
            'taxi': 13.89,  # 50 km/h
            'unknown': 13.89
        }.get(mode.lower(), 6.94)

        # Check if the trip's maximum speed exceeds the mode-based cap
        if trip_max_speed >= speed_cap_mps:
            print(f"Trip max speed ({trip_max_speed:.2f} m/s) exceeds the cap of {speed_cap_mps:.2f} m/s.")
        else:
            print(f"Trip max speed ({trip_max_speed:.2f} m/s) is within the cap of {speed_cap_mps:.2f} m/s.")

        # Calculate rows counts before filtering
        initial_rows = len(gdf_clean)
        
        # Count extreme outliers (speed > 2x cap)
        extreme_outliers = (gdf_clean['speed'] > 2 * speed_cap_mps).sum()
        extreme_outlier_rows_global += extreme_outliers
        
        # Count very extreme outliers (speed > 5x cap)
        very_extreme_outliers = (gdf_clean['speed'] > 5 * speed_cap_mps).sum()
        very_extreme_outlier_rows_global += very_extreme_outliers
        
        # Count regular capped points (cap < speed <= 2*cap)
        regular_capped = ((gdf_clean['speed'] > speed_cap_mps) & (gdf_clean['speed'] <= 2 * speed_cap_mps)).sum()
        regular_capped_rows_global += regular_capped
        
        print(f"Initial rows: {initial_rows}")
        print(f"Speed cap: {speed_cap_mps}")
        print(f"Regular capped (cap < speed <= 2x cap): {regular_capped}")
        print(f"Extreme outliers (>2x cap): {extreme_outliers}")
        print(f"Very extreme outliers (>5x cap): {very_extreme_outliers}")
        
        # REMOVE extreme outliers (speed > 2x cap)
        gdf_clean = gdf_clean[gdf_clean['speed'] <= 2 * speed_cap_mps].copy()
        
        # Apply capping for the remaining rows that exceed cap but aren't extreme
        gdf_clean['speed'] = gdf_clean['speed'].clip(upper=speed_cap_mps)
        
        # Update global counter for total rows (after removal)
        total_rows_global += len(gdf_clean)
        
        print(f"Rows after removal: {len(gdf_clean)}")
        print(f"Points removed: {initial_rows - len(gdf_clean)}")
        print(f"Total rows processed globally: {total_rows_global}")

        # Compute enriched metrics (this will perform the capping inside the function)
        metrics = compute_enriched_metrics(gdf_clean)
        if metrics is None:
            print(f"Metrics could not be computed for file {file}. Skipping.")
            continue

        # Calculate additional trip-level information.
        if len(gdf_clean) > 0:  # Check if we still have points after removal
            start_time = gdf_clean['datetime'].iloc[0]
            end_time = gdf_clean['datetime'].iloc[-1]
            duration_sec = (end_time - start_time).total_seconds()

            if 'transport_mode' in gdf_clean.columns:
                mode_series = gdf_clean['transport_mode'].dropna()
                if len(mode_series.unique()) == 1:
                    trip_mode = mode_series.iloc[0]
                else:
                    trip_mode = mode_series.mode()[0]
            else:
                trip_mode = "Unknown"

            summary, _ = generate_trip_description(gdf_clean, metrics)

            # Append additional information to metrics.
            metrics['trip_id'] = os.path.basename(file)
            metrics['start_time'] = start_time
            metrics['end_time'] = end_time
            metrics['duration_sec'] = duration_sec
            metrics['transport_mode'] = trip_mode
            
            # Add outlier info to the metrics
            metrics['initial_points'] = initial_rows
            metrics['points_removed'] = initial_rows - len(gdf_clean)
            metrics['pct_points_removed'] = ((initial_rows - len(gdf_clean)) / initial_rows) * 100 if initial_rows > 0 else 0
            metrics['regular_capped_points'] = regular_capped
            metrics['extreme_outliers_removed'] = extreme_outliers
            metrics['speed_cap'] = speed_cap_mps
            
            metrics['trip_summary'] = summary

            trip_data.append(metrics)

    except Exception as e:
        print(f"Error processing file {file}: {e}")

# Create a DataFrame from the aggregated trip metrics.
df_trip_level = pd.DataFrame(trip_data)
print("Trip-level dataset shape:", df_trip_level.shape)
display(df_trip_level.head())

# Save the aggregated trip-level dataset to CSV.
output_csv = "trip_level_data.csv"
df_trip_level.to_csv(output_csv, index=False)
print(f"Trip-level data saved to '{output_csv}'.")

# Compute global percentages
total_initial_rows = total_rows_global + extreme_outlier_rows_global
if total_initial_rows > 0:
    pct_extreme_outliers_global = (extreme_outlier_rows_global / total_initial_rows) * 100
    pct_very_extreme_outliers_global = (very_extreme_outlier_rows_global / total_initial_rows) * 100
    pct_regular_capped_global = (regular_capped_rows_global / total_initial_rows) * 100
else:
    pct_extreme_outliers_global = 0
    pct_very_extreme_outliers_global = 0
    pct_regular_capped_global = 0

# Write the analysis to a text file in the ML_result folder.
output_analysis_path = os.path.join("ML_result", "capping_analysis.txt")
os.makedirs(os.path.dirname(output_analysis_path), exist_ok=True)  # Ensure the folder exists

with open(output_analysis_path, "w") as f:
    f.write("Global Cleaning Analysis\n")
    f.write("=======================\n")
    f.write(f"Total initial rows: {total_initial_rows}\n")
    f.write(f"Extreme outliers removed (>2x cap): {extreme_outlier_rows_global}\n")
    f.write(f"Percentage of extreme outliers removed: {pct_extreme_outliers_global:.2f}%\n")
    f.write(f"Very extreme outliers subset (>5x cap): {very_extreme_outlier_rows_global}\n")
    f.write(f"Percentage of very extreme outliers: {pct_very_extreme_outliers_global:.2f}%\n\n")
    
    f.write("Remaining Data Analysis\n")
    f.write("=======================\n")
    f.write(f"Total rows after removal: {total_rows_global}\n")
    f.write(f"Rows with speed capped (speed > cap but ≤ 2x cap): {regular_capped_rows_global}\n")
    f.write(f"Percentage of rows capped: {pct_regular_capped_global:.2f}%\n")

print(f"Enhanced cleaning analysis saved to '{output_analysis_path}'.")

Skipping field time: unsupported OGR type: 10


Number of GeoJSON files found: 6934
Data retained: 674 rows after cleaning.
Trip max speed (292.09 m/s) exceeds the cap of 2.78 m/s.
Initial rows: 673
Speed cap: 2.78
Regular capped (cap < speed <= 2x cap): 8
Extreme outliers (>2x cap): 42
Very extreme outliers (>5x cap): 35
Rows after removal: 630
Points removed: 43
Total rows processed globally: 630
Outlier count before clipping: 53


Skipping field time: unsupported OGR type: 10
Skipping field time: unsupported OGR type: 10


Data discarded: mean speed too high (34.69 m/s).
File Sub_Trajectories_Cleaned/20081101040025/train_seg1_cleaned.geojson did not pass cleaning. Skipping.
Data retained: 291 rows after cleaning.
Trip max speed (42.41 m/s) exceeds the cap of 2.78 m/s.
Initial rows: 290
Speed cap: 2.78
Regular capped (cap < speed <= 2x cap): 13
Extreme outliers (>2x cap): 9
Very extreme outliers (>5x cap): 1
Rows after removal: 280
Points removed: 10
Total rows processed globally: 910
Outlier count before clipping: 21


Skipping field time: unsupported OGR type: 10


...

Skipping field time: unsupported OGR type: 10


Data retained: 41 rows after cleaning.
Trip max speed (50.86 m/s) exceeds the cap of 6.94 m/s.
Initial rows: 40
Speed cap: 6.94
Regular capped (cap < speed <= 2x cap): 8
Extreme outliers (>2x cap): 18
Very extreme outliers (>5x cap): 1
Rows after removal: 21
Points removed: 19
Total rows processed globally: 185989
Outlier count before clipping: 10


Skipping field time: unsupported OGR type: 10


Data retained: 282 rows after cleaning.
Trip max speed (115.36 m/s) exceeds the cap of 11.11 m/s.
Initial rows: 281
Speed cap: 11.11
Regular capped (cap < speed <= 2x cap): 7
Extreme outliers (>2x cap): 6
Very extreme outliers (>5x cap): 3
Rows after removal: 274
Points removed: 7
Total rows processed globally: 186263
Outlier count before clipping: 22


Skipping field time: unsupported OGR type: 10


In [6]:
import pandas as pd
import re
import numpy as np

# Main function to process the CSV file
def process_csv(input_file, output_file):
    try:
        # Read the CSV file with the correct handling for the trip_summary field
        # which contains newlines
        df = pd.read_csv(input_file, quotechar='"', escapechar='\\')
        
        # Count rows before processing
        total_rows = len(df)
        modified_rows = 0
        
        # First, calculate statistics from non-zero values to use for imputation
        avg_altitude_median = df[df['avg_altitude'] != 0]['avg_altitude'].median()
        total_gain_median = df[df['total_altitude_gain'] != 0]['total_altitude_gain'].median()
        total_loss_median = df[df['total_altitude_loss'] != 0]['total_altitude_loss'].median()
        vert_accel_median = df[df['avg_vertical_acceleration'] != 0]['avg_vertical_acceleration'].median()
        
        print(f"Calculated medians from non-zero values:")
        print(f"- Average Altitude: {avg_altitude_median:.2f} m")
        print(f"- Total Altitude Gain: {total_gain_median:.2f} m")
        print(f"- Total Altitude Loss: {total_loss_median:.2f} m")
        print(f"- Average Vertical Acceleration: {vert_accel_median:.4f} m/s²")
        
        # Create a new column to flag which rows had zero values (useful for ML)
        df['altitude_data_missing'] = ((df['avg_altitude'] == 0) | 
                                     (df['total_altitude_gain'] == 0) | 
                                     (df['total_altitude_loss'] == 0) | 
                                     (df['avg_vertical_acceleration'] == 0)).astype(int)
        
        # Create a deep copy for the NaN version
        df_nan = df.copy(deep=True)
        
        # Create a deep copy for the imputed version
        df_imputed = df.copy(deep=True)
        
        print("Processing rows with zero altitude values...")
        
        # Process each row in NaN version
        for index, row in df_nan.iterrows():
            modified = False
            
            # Replace zeros with NaN in the NaN version
            if row['avg_altitude'] == 0.0:
                df_nan.at[index, 'avg_altitude'] = np.nan
                modified = True
                
            if row['total_altitude_gain'] == 0.0:
                df_nan.at[index, 'total_altitude_gain'] = np.nan
                modified = True
                
            if row['total_altitude_loss'] == 0.0:
                df_nan.at[index, 'total_altitude_loss'] = np.nan
                modified = True
                
            if row['avg_vertical_acceleration'] == 0.0:
                df_nan.at[index, 'avg_vertical_acceleration'] = np.nan
                modified = True
            
            # Update the trip_summary by marking the missing values
            if modified:
                modified_rows += 1
                # Replace zeros in trip_summary with "Missing" indicator
                summary = row['trip_summary']
                summary = re.sub(r'- Average Altitude: 0\.00 m', 
                                 '- Average Altitude: Missing', summary)
                summary = re.sub(r'- Total Altitude Gain: 0\.00 m', 
                                 '- Total Altitude Gain: Missing', summary)
                summary = re.sub(r'- Total Altitude Loss: 0\.00 m', 
                                 '- Total Altitude Loss: Missing', summary)
                summary = re.sub(r'- Average Vertical Acceleration: 0\.00 m/s²', 
                                 '- Average Vertical Acceleration: Missing', summary)
                
                df_nan.at[index, 'trip_summary'] = summary
        
        # Process each row in imputed version - replace with calculated medians
        for index, row in df_imputed.iterrows():
            modified = False
            new_avg_alt = row['avg_altitude']
            new_alt_gain = row['total_altitude_gain']
            new_alt_loss = row['total_altitude_loss']
            new_vert_accel = row['avg_vertical_acceleration']
            
            # Replace zeros with median values in the imputed version
            if row['avg_altitude'] == 0.0:
                new_avg_alt = avg_altitude_median
                df_imputed.at[index, 'avg_altitude'] = avg_altitude_median
                modified = True
                
            if row['total_altitude_gain'] == 0.0:
                new_alt_gain = total_gain_median
                df_imputed.at[index, 'total_altitude_gain'] = total_gain_median
                modified = True
                
            if row['total_altitude_loss'] == 0.0:
                new_alt_loss = total_loss_median
                df_imputed.at[index, 'total_altitude_loss'] = total_loss_median
                modified = True
                
            if row['avg_vertical_acceleration'] == 0.0:
                new_vert_accel = vert_accel_median
                df_imputed.at[index, 'avg_vertical_acceleration'] = vert_accel_median
                modified = True
            
            # Update the trip_summary in imputed version with the median values
            if modified:
                summary = row['trip_summary']
                
                # Replace values in the trip summary text
                if new_avg_alt != 0:
                    summary = re.sub(r'- Average Altitude: 0\.00 m', 
                                    f'- Average Altitude: {new_avg_alt:.2f} m', summary)
                
                if new_alt_gain != 0:
                    summary = re.sub(r'- Total Altitude Gain: 0\.00 m', 
                                    f'- Total Altitude Gain: {new_alt_gain:.2f} m', summary)
                
                if new_alt_loss != 0:
                    summary = re.sub(r'- Total Altitude Loss: 0\.00 m', 
                                    f'- Total Altitude Loss: {new_alt_loss:.2f} m', summary)
                
                if new_vert_accel != 0:
                    summary = re.sub(r'- Average Vertical Acceleration: 0\.00 m/s²', 
                                    f'- Average Vertical Acceleration: {new_vert_accel:.2f} m/s²', summary)
                
                df_imputed.at[index, 'trip_summary'] = summary
                
        # Reorder columns so that 'altitude_data_missing' appears before 'trip_summary'
        def reorder_columns(df):
            cols = list(df.columns)
            if 'trip_summary' in cols and 'altitude_data_missing' in cols:
                cols.remove('altitude_data_missing')
                idx = cols.index('trip_summary')
                cols.insert(idx, 'altitude_data_missing')
            return df[cols]
        
        df_nan = reorder_columns(df_nan)
        df_imputed = reorder_columns(df_imputed)
        
        # Save both versions to different files
        nan_output_file = output_file
        imputed_output_file = output_file.replace('.csv', '_imputed.csv')
        
        df_nan.to_csv(nan_output_file, index=False, quotechar='"', escapechar='\\')
        df_imputed.to_csv(imputed_output_file, index=False, quotechar='"', escapechar='\\')
        
        print(f"Processing complete. Modified {modified_rows} out of {total_rows} rows.")
        print(f"Created two output files:")
        print(f"1. {nan_output_file} - with zeros converted to NaN (better for ML)")
        print(f"2. {imputed_output_file} - with values imputed using medians from non-zero data")
        print(f"Added 'altitude_data_missing' column to flag rows with missing altitude data")
        
        return True
        
    except Exception as e:
        print(f"Error processing the CSV file: {e}")
        return False

# Call the function
if __name__ == "__main__":
    input_file = "trip_level_data.csv"
    output_file = "trip_level_data_processed.csv"
    process_csv(input_file, output_file)

Calculated medians from non-zero values:
- Average Altitude: 51.00 m
- Total Altitude Gain: 50.30 m
- Total Altitude Loss: 50.70 m
- Average Vertical Acceleration: 0.0309 m/s²
Processing rows with zero altitude values...
Processing complete. Modified 1459 out of 7120 rows.
Created two output files:
1. trip_level_data_processed.csv - with zeros converted to NaN (better for ML)
2. trip_level_data_processed_imputed.csv - with values imputed using medians from non-zero data
Added 'altitude_data_missing' column to flag rows with missing altitude data
